# Dynamical Quantum Phase Transition — Loschmidt Echo Cusps

> 🚀 **Try Maestro GPU mode with a free trial.**
> Sign up at **[maestro.qoroquantum.net](https://maestro.qoroquantum.net)** — no credit card required.

## Why GPU Mode?

DQPT simulations sweep **multiple quench strengths × 40 Trotter steps** each. Crisp cusps need χ=64+ on 80 qubits. GPU mode makes the full sweep practical.

## Step 0 — Setup

```bash
pip install qoro-maestro matplotlib numpy
```

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import maestro

from dqpt_demo import (
    build_quench_circuit,
    build_x_observables,
    compute_loschmidt_rate,
    exact_loschmidt_rate_tfim,
)

---

## Phase 1 — Local (CPU, 30 qubits)

30 qubits, χ=32, 3 quench strengths × 20 Trotter steps. See the cusps emerge at modest system size.

In [ ]:
N_CPU = 30
CHI_CPU = 32
N_STEPS_CPU = 20
DT = 0.1
J = 1.0
H_VALUES = [0.2, 0.5, 0.8]  # All subcritical → DQPTs expected

print(f"Phase 1: {N_CPU} qubits, χ={CHI_CPU}, {N_STEPS_CPU} steps (CPU)")
print(f"Quench values: h_f = {H_VALUES}")
print(f"Critical point: h_c = J = {J}")

In [ ]:
x_obs_cpu = build_x_observables(N_CPU)
results_cpu = []

t0_total = time.time()
for h_f in H_VALUES:
    print(f"\n  Quench h_f = {h_f:.1f} ...")
    data = {'h_f': h_f, 'times': [0.0], 'rate': [0.0], 'avg_x': [1.0]}
    
    for step in range(1, N_STEPS_CPU + 1):
        qc = build_quench_circuit(N_CPU, J, h_f, DT, step)
        res = qc.estimate(
            simulator_type=maestro.SimulatorType.QCSim,
            simulation_type=maestro.SimulationType.MatrixProductState,
            observables=x_obs_cpu,
            max_bond_dimension=CHI_CPU,
        )
        rate, avg_x = compute_loschmidt_rate(res['expectation_values'], N_CPU)
        data['times'].append(step * DT)
        data['rate'].append(rate)
        data['avg_x'].append(avg_x)
    
    results_cpu.append(data)
    print(f"  Done — λ_max = {max(data['rate']):.4f}")

cpu_time = time.time() - t0_total
print(f"\n✅ Phase 1 complete in {cpu_time:.1f}s")

In [ ]:
colors = ['#E91E63', '#1565C0', '#4CAF50']

fig, ax = plt.subplots(figsize=(10, 6))
for i, data in enumerate(results_cpu):
    ax.plot(data['times'], data['rate'], 'o-', color=colors[i],
            linewidth=2, markersize=4, label=f'h_f/J = {data["h_f"]:.1f}')

# Exact reference
t_dense = np.linspace(0, N_STEPS_CPU * DT, 200)
for i, h_f in enumerate(H_VALUES):
    exact_r = exact_loschmidt_rate_tfim(J, h_f, t_dense)
    ax.plot(t_dense, exact_r, '--', color=colors[i], alpha=0.4, linewidth=1.5)

ax.set_xlabel('Time t', fontsize=13)
ax.set_ylabel('Loschmidt Rate Function λ(t)', fontsize=13)
ax.set_title(f'Phase 1 (CPU): {N_CPU} qubits, χ={CHI_CPU}\n'
             f'Completed in {cpu_time:.1f}s', fontsize=14)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Phase 2 — GPU Mode (80 qubits, higher χ)

80 qubits, χ=64, 40 Trotter steps. Sharper cusps, larger system.

👉 **[Start your free GPU trial →](https://maestro.qoroquantum.net)**

In [ ]:
N_GPU = 80
CHI_GPU = 64
N_STEPS_GPU = 40

print(f"Phase 2: {N_GPU} qubits, χ={CHI_GPU}, {N_STEPS_GPU} steps (GPU)")

x_obs_gpu = build_x_observables(N_GPU)
results_gpu = []

t0_total = time.time()
for h_f in H_VALUES:
    print(f"\n  Quench h_f = {h_f:.1f} ...")
    data = {'h_f': h_f, 'times': [0.0], 'rate': [0.0], 'avg_x': [1.0]}
    
    for step in range(1, N_STEPS_GPU + 1):
        qc = build_quench_circuit(N_GPU, J, h_f, DT, step)
        res = qc.estimate(
            simulator_type=maestro.SimulatorType.CuQuantum,
            simulation_type=maestro.SimulationType.MatrixProductState,
            observables=x_obs_gpu,
            max_bond_dimension=CHI_GPU,
        )
        rate, avg_x = compute_loschmidt_rate(res['expectation_values'], N_GPU)
        data['times'].append(step * DT)
        data['rate'].append(rate)
        data['avg_x'].append(avg_x)
        if step % 10 == 0:
            print(f"    step {step}  λ={rate:.4f}")
    
    results_gpu.append(data)

gpu_time = time.time() - t0_total
print(f"\n✅ Phase 2 complete in {gpu_time:.1f}s")
print(f"\n⚡ Phase 1 (CPU, {N_CPU}q): {cpu_time:.1f}s")
print(f"⚡ Phase 2 (GPU, {N_GPU}q): {gpu_time:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Phase 1
for i, data in enumerate(results_cpu):
    axes[0].plot(data['times'], data['rate'], 'o-', color=colors[i],
                linewidth=2, markersize=3, label=f'h_f/J = {data["h_f"]:.1f}')
axes[0].set_title(f'Phase 1 — CPU: {N_CPU}q, χ={CHI_CPU}\n{cpu_time:.1f}s')
axes[0].set_xlabel('Time t')
axes[0].set_ylabel('λ(t)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Phase 2
t_dense2 = np.linspace(0, N_STEPS_GPU * DT, 300)
for i, data in enumerate(results_gpu):
    axes[1].plot(data['times'], data['rate'], 'o', color=colors[i],
                markersize=3, alpha=0.8, label=f'MPS (h_f/J={data["h_f"]:.1f})')
    exact_r = exact_loschmidt_rate_tfim(J, data['h_f'], t_dense2)
    axes[1].plot(t_dense2, exact_r, '-', color=colors[i], alpha=0.4, linewidth=1.5)

axes[1].set_title(f'Phase 2 — GPU: {N_GPU}q, χ={CHI_GPU}\n{gpu_time:.1f}s')
axes[1].set_xlabel('Time t')
axes[1].set_ylabel('λ(t)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dqpt_loschmidt.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Ready for Large-Scale Quench Dynamics?

GPU mode makes 80+ qubit DQPT sweeps practical — sharper cusps, larger systems, reasonable runtime.

👉 **[Start your free GPU trial at maestro.qoroquantum.net](https://maestro.qoroquantum.net)**